# PriceFusion V0.4 — Advanced Structured Feature Engineering

**Project:** PriceFusion — Multimodal Product Price Intelligence  
**Stage:** V0.4  
**Goal:** Build a robust reusable feature-engineering pipeline from `catalog_content` for use in gradient-boosted and hybrid models.

This notebook extracts and engineers:
- Item name
- Raw `Value` and `Unit`
- Normalized unit categories
- Pack, case, set, count, serving, pallet, bottle, can, bag and piece quantities
- Weight and volume measurements
- Total inferred weight/volume where possible
- Bulk/case/pallet indicators
- Text length and numeric-density features
- Unit-price interaction features
- Reusable train/validation/test feature tables aligned with the permanent V0.3 split


## 1. Imports

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 220)

print('V0.4 environment ready.')

V0.4 environment ready.


## 2. Resolve Project Paths

In [2]:
cwd = Path.cwd().resolve()

if (cwd / 'dataset').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'dataset').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        'Could not find dataset/. Launch Jupyter from PriceFusion/ or PriceFusion/notebooks/.'
    )

DATASET_DIR = PROJECT_ROOT / 'dataset'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
SPLITS_DIR = ARTIFACTS_DIR / 'splits'
FEATURES_DIR = ARTIFACTS_DIR / 'features'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
FIGURES_DIR = ARTIFACTS_DIR / 'figures'

for p in [FEATURES_DIR, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATASET_DIR / 'train.csv'
TEST_CSV = DATASET_DIR / 'test.csv'
TRAIN_IDS_PATH = SPLITS_DIR / 'train_ids.csv'
VALID_IDS_PATH = SPLITS_DIR / 'validation_ids.csv'

print('Project root :', PROJECT_ROOT)
print('Features dir :', FEATURES_DIR)
print('Splits dir   :', SPLITS_DIR)

Project root : /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion
Features dir : /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion/artifacts/features
Splits dir   : /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion/artifacts/splits


## 3. Load Raw Train/Test and Saved V0.3 Split

In [3]:
train_raw = pd.read_csv(TRAIN_CSV)
test_raw = pd.read_csv(TEST_CSV)

assert TRAIN_IDS_PATH.exists(), 'Missing artifacts/splits/train_ids.csv from V0.3'
assert VALID_IDS_PATH.exists(), 'Missing artifacts/splits/validation_ids.csv from V0.3'

train_ids = pd.read_csv(TRAIN_IDS_PATH)['sample_id']
valid_ids = pd.read_csv(VALID_IDS_PATH)['sample_id']

print('Raw train shape:', train_raw.shape)
print('Raw test shape :', test_raw.shape)
print('Saved train IDs:', len(train_ids))
print('Saved valid IDs:', len(valid_ids))

Raw train shape: (75000, 4)
Raw test shape : (75000, 3)
Saved train IDs: 60000
Saved valid IDs: 15000


## 4. Core Text Helpers

These helpers are deliberately deterministic so the same parsing can be reused later in production inference.

In [4]:
def safe_text(x):
    return '' if pd.isna(x) else str(x)

def extract_line_value(text, label):
    pattern = rf'(?im)^{re.escape(label)}\s*:\s*([^\n\r]+)'
    m = re.search(pattern, safe_text(text))
    return m.group(1).strip() if m else None

def parse_float(value):
    if value is None:
        return np.nan
    value = str(value).strip()
    if value.lower() in {'', 'nan', 'none', 'null'}:
        return np.nan
    value = value.replace(',', '')
    try:
        return float(value)
    except ValueError:
        return np.nan

def first_number(patterns, text, min_value=0, max_value=None):
    text = safe_text(text)
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            try:
                val = float(m.group(1))
                if val < min_value:
                    continue
                if max_value is not None and val > max_value:
                    continue
                return val
            except Exception:
                pass
    return np.nan

## 5. Normalize Units

In [5]:
UNIT_MAP = {
    'oz': 'ounce',
    'ounce': 'ounce',
    'ounces': 'ounce',
    'fl oz': 'fl_oz',
    'fl. oz': 'fl_oz',
    'fl. oz.': 'fl_oz',
    'fluid ounce': 'fl_oz',
    'fluid ounces': 'fl_oz',
    'fluid ounce(s)': 'fl_oz',
    'lb': 'pound',
    'lbs': 'pound',
    'pound': 'pound',
    'pounds': 'pound',
    'g': 'gram',
    'gr': 'gram',
    'gram': 'gram',
    'grams': 'gram',
    'gramm': 'gram',
    'kg': 'kilogram',
    'kilogram': 'kilogram',
    'kilograms': 'kilogram',
    'ml': 'milliliter',
    'milliliter': 'milliliter',
    'milliliters': 'milliliter',
    'millilitre': 'milliliter',
    'millilitres': 'milliliter',
    'l': 'liter',
    'liter': 'liter',
    'liters': 'liter',
    'litre': 'liter',
    'litres': 'liter',
    'count': 'count',
    'counts': 'count',
    'ct': 'count',
    'each': 'count',
    'piece': 'count',
    'pieces': 'count',
    'pack': 'count',
    'packs': 'count',
    'bottle': 'count',
    'bottles': 'count',
    'can': 'count',
    'cans': 'count',
    'bag': 'count',
    'bags': 'count',
}

def normalize_unit(unit):
    if unit is None or pd.isna(unit):
        return 'unknown'
    u = str(unit).strip().lower()
    u = re.sub(r'\s+', ' ', u)
    return UNIT_MAP.get(u, u if u else 'unknown')

## 6. Quantity Parsers

In [6]:
PACK_PATTERNS = [
    r'pack\s+of\s+(\d+)',
    r'(\d+)\s*[- ]?pack\b',
    r'pk[- ]?\s*(\d+)\b',
    r'\bpack\s*(\d+)\b',
]

CASE_PATTERNS = [
    r'(\d+)\s+per\s+case\b',
    r'case\s+of\s+(\d+)',
    r'(\d+)\s*[- ]?case\b',
]

SET_PATTERNS = [
    r'set\s+of\s+(\d+)',
    r'(\d+)\s*[- ]?piece\s+set\b',
]

SERVING_PATTERNS = [
    r'(\d[\d,]*)\s+servings?\b',
]

PALLET_PATTERNS = [
    r'pallet\s+of\s+(\d+)\s+cases?\b',
    r'(\d+)\s+cases?\s+(?:on|per)\s+pallet\b',
]

COUNT_PATTERNS = [
    r'\b(\d+)\s*[- ]?count\b',
    r'\b(\d+)\s*ct\b',
    r'\b(\d+)\s*pcs?\b',
    r'\b(\d+)\s*pieces?\b',
    r'\b(\d+)\s*bottles?\b',
    r'\b(\d+)\s*cans?\b',
    r'\b(\d+)\s*bags?\b',
    r'\b(\d+)\s*packets?\b',
]

def extract_pack_quantity(text):
    v = first_number(PACK_PATTERNS, text, min_value=1, max_value=1000)
    return int(v) if not np.isnan(v) else 1

def extract_case_quantity(text):
    v = first_number(CASE_PATTERNS, text, min_value=1, max_value=5000)
    return int(v) if not np.isnan(v) else 1

def extract_set_quantity(text):
    v = first_number(SET_PATTERNS, text, min_value=1, max_value=5000)
    return int(v) if not np.isnan(v) else 1

def extract_servings(text):
    clean = safe_text(text).replace(',', '')
    v = first_number(SERVING_PATTERNS, clean, min_value=1, max_value=100000)
    return v

def extract_pallet_cases(text):
    v = first_number(PALLET_PATTERNS, text, min_value=1, max_value=10000)
    return v

def extract_explicit_count(text):
    v = first_number(COUNT_PATTERNS, text, min_value=1, max_value=100000)
    return v

## 7. Measurement Parsers

In [7]:
MEASUREMENT_PATTERNS = {
    'ounce': [r'(\d+(?:\.\d+)?)\s*(?:ounce|ounces|oz)\b'],
    'fl_oz': [r'(\d+(?:\.\d+)?)\s*(?:fl\.?\s*oz\.?|fluid\s+ounces?|fluid\s+ounce)\b'],
    'pound': [r'(\d+(?:\.\d+)?)\s*(?:pounds?|lbs?|lb)\b'],
    'gram': [r'(\d+(?:\.\d+)?)\s*(?:grams?|gramm|g)\b'],
    'kilogram': [r'(\d+(?:\.\d+)?)\s*(?:kilograms?|kg)\b'],
    'milliliter': [r'(\d+(?:\.\d+)?)\s*(?:milliliters?|millilitres?|ml)\b'],
    'liter': [r'(\d+(?:\.\d+)?)\s*(?:liters?|litres?|l)\b'],
}

def extract_measurement(text, unit_key):
    return first_number(MEASUREMENT_PATTERNS[unit_key], text, min_value=0, max_value=1000000)

def oz_to_grams(x):
    return x * 28.349523125

def lb_to_grams(x):
    return x * 453.59237

def kg_to_grams(x):
    return x * 1000.0

def fl_oz_to_ml(x):
    return x * 29.5735295625

def liter_to_ml(x):
    return x * 1000.0

## 8. Main Feature Engineering Function

In [8]:
def engineer_features(df):
    out = pd.DataFrame(index=df.index)
    out['sample_id'] = df['sample_id'].values

    text = df['catalog_content'].fillna('').astype(str)

    # Base structured fields
    out['item_name'] = text.map(lambda x: extract_line_value(x, 'Item Name'))
    out['raw_value'] = text.map(lambda x: parse_float(extract_line_value(x, 'Value')))
    out['raw_unit'] = text.map(lambda x: extract_line_value(x, 'Unit'))
    out['unit_normalized'] = out['raw_unit'].map(normalize_unit)

    # Text features
    out['catalog_char_count'] = text.str.len()
    out['catalog_word_count'] = text.str.split().str.len()
    out['catalog_line_count'] = text.str.count(r'\n') + 1
    out['catalog_digit_count'] = text.str.count(r'\d')
    out['numeric_token_count'] = text.str.count(r'\b\d+(?:\.\d+)?\b')
    out['bullet_count'] = text.str.count(r'(?i)Bullet Point')
    out['uppercase_count'] = text.str.count(r'\b[A-Z]{2,}\b')
    out['special_char_count'] = text.str.count(r'[^A-Za-z0-9\s]')

    # Quantity features
    out['pack_quantity'] = text.map(extract_pack_quantity)
    out['case_quantity'] = text.map(extract_case_quantity)
    out['set_quantity'] = text.map(extract_set_quantity)
    out['servings'] = text.map(extract_servings)
    out['pallet_cases'] = text.map(extract_pallet_cases)
    out['explicit_count'] = text.map(extract_explicit_count)

    # Semantic flags
    lower = text.str.lower()
    out['has_pack'] = lower.str.contains(r'\bpack\b', regex=True).astype('int8')
    out['has_case'] = lower.str.contains(r'\bcase\b', regex=True).astype('int8')
    out['has_pallet'] = lower.str.contains(r'\bpallet\b', regex=True).astype('int8')
    out['has_bulk'] = lower.str.contains(r'\bbulk\b', regex=True).astype('int8')
    out['has_serving'] = lower.str.contains(r'\bservings?\b', regex=True).astype('int8')
    out['has_bottle'] = lower.str.contains(r'\bbottles?\b', regex=True).astype('int8')
    out['has_can'] = lower.str.contains(r'\bcans?\b', regex=True).astype('int8')
    out['has_bag'] = lower.str.contains(r'\bbags?\b', regex=True).astype('int8')
    out['has_frozen'] = lower.str.contains(r'\bfrozen\b', regex=True).astype('int8')
    out['has_organic'] = lower.str.contains(r'\borganic\b', regex=True).astype('int8')

    # Measurements from natural text
    out['measurement_ounce'] = text.map(lambda x: extract_measurement(x, 'ounce'))
    out['measurement_fl_oz'] = text.map(lambda x: extract_measurement(x, 'fl_oz'))
    out['measurement_pound'] = text.map(lambda x: extract_measurement(x, 'pound'))
    out['measurement_gram'] = text.map(lambda x: extract_measurement(x, 'gram'))
    out['measurement_kilogram'] = text.map(lambda x: extract_measurement(x, 'kilogram'))
    out['measurement_ml'] = text.map(lambda x: extract_measurement(x, 'milliliter'))
    out['measurement_liter'] = text.map(lambda x: extract_measurement(x, 'liter'))

    # Convert explicit raw Value+Unit into unified measures
    out['raw_weight_grams'] = np.nan
    out['raw_volume_ml'] = np.nan

    mask = out['unit_normalized'].eq('ounce')
    out.loc[mask, 'raw_weight_grams'] = out.loc[mask, 'raw_value'].map(oz_to_grams)

    mask = out['unit_normalized'].eq('pound')
    out.loc[mask, 'raw_weight_grams'] = out.loc[mask, 'raw_value'].map(lb_to_grams)

    mask = out['unit_normalized'].eq('gram')
    out.loc[mask, 'raw_weight_grams'] = out.loc[mask, 'raw_value']

    mask = out['unit_normalized'].eq('kilogram')
    out.loc[mask, 'raw_weight_grams'] = out.loc[mask, 'raw_value'].map(kg_to_grams)

    mask = out['unit_normalized'].eq('fl_oz')
    out.loc[mask, 'raw_volume_ml'] = out.loc[mask, 'raw_value'].map(fl_oz_to_ml)

    mask = out['unit_normalized'].eq('milliliter')
    out.loc[mask, 'raw_volume_ml'] = out.loc[mask, 'raw_value']

    mask = out['unit_normalized'].eq('liter')
    out.loc[mask, 'raw_volume_ml'] = out.loc[mask, 'raw_value'].map(liter_to_ml)

    # Best parsed natural-text measurement in unified units
    parsed_weight_candidates = pd.DataFrame({
        'oz': out['measurement_ounce'].map(oz_to_grams),
        'lb': out['measurement_pound'].map(lb_to_grams),
        'g': out['measurement_gram'],
        'kg': out['measurement_kilogram'].map(kg_to_grams),
    })
    out['parsed_weight_grams'] = parsed_weight_candidates.bfill(axis=1).iloc[:, 0]

    parsed_volume_candidates = pd.DataFrame({
        'fl_oz': out['measurement_fl_oz'].map(fl_oz_to_ml),
        'ml': out['measurement_ml'],
        'liter': out['measurement_liter'].map(liter_to_ml),
    })
    out['parsed_volume_ml'] = parsed_volume_candidates.bfill(axis=1).iloc[:, 0]

    # Multipliers
    out['quantity_multiplier'] = (
        out['pack_quantity'].fillna(1).clip(lower=1) *
        out['case_quantity'].fillna(1).clip(lower=1) *
        out['set_quantity'].fillna(1).clip(lower=1)
    )

    out['total_parsed_weight_grams'] = out['parsed_weight_grams'] * out['quantity_multiplier']
    out['total_parsed_volume_ml'] = out['parsed_volume_ml'] * out['quantity_multiplier']

    # Conservative total-unit signal using explicit count when available
    out['effective_count'] = pd.concat([
        out['explicit_count'],
        out['servings'],
        out['pallet_cases'],
    ], axis=1).max(axis=1, skipna=True)
    out['effective_count'] = out['effective_count'].fillna(out['quantity_multiplier'])

    # Log transforms for highly skewed numerics
    log_cols = [
        'raw_value', 'catalog_char_count', 'catalog_word_count',
        'numeric_token_count', 'pack_quantity', 'case_quantity',
        'servings', 'pallet_cases', 'explicit_count',
        'raw_weight_grams', 'raw_volume_ml',
        'parsed_weight_grams', 'parsed_volume_ml',
        'total_parsed_weight_grams', 'total_parsed_volume_ml',
        'effective_count',
    ]

    for col in log_cols:
        out[f'log1p_{col}'] = np.log1p(out[col].clip(lower=0))

    return out

## 9. Engineer Train and Test Features

In [9]:
train_features_all = engineer_features(train_raw)
test_features = engineer_features(test_raw)

train_features_all['price'] = train_raw['price'].values

print('Train engineered shape:', train_features_all.shape)
print('Test engineered shape :', test_features.shape)
display(train_features_all.head())

Train engineered shape: (75000, 61)
Test engineered shape : (75000, 60)


,sample_id,item_name,raw_value,raw_unit,unit_normalized,catalog_char_count,catalog_word_count,catalog_line_count,catalog_digit_count,numeric_token_count,bullet_count,uppercase_count,special_char_count,pack_quantity,case_quantity,set_quantity,servings,pallet_cases,explicit_count,has_pack,has_case,has_pallet,has_bulk,has_serving,has_bottle,has_can,has_bag,has_frozen,has_organic,measurement_ounce,measurement_fl_oz,measurement_pound,measurement_gram,measurement_kilogram,measurement_ml,measurement_liter,raw_weight_grams,raw_volume_ml,parsed_weight_grams,parsed_volume_ml,quantity_multiplier,total_parsed_weight_grams,total_parsed_volume_ml,effective_count,log1p_raw_value,log1p_catalog_char_count,log1p_catalog_word_count,log1p_numeric_token_count,log1p_pack_quantity,log1p_case_quantity,log1p_servings,log1p_pallet_cases,log1p_explicit_count,log1p_raw_weight_grams,log1p_raw_volume_ml,log1p_parsed_weight_grams,log1p_parsed_volume_ml,log1p_total_parsed_weight_grams,log1p_total_parsed_volume_ml,log1p_effective_count,price
0,33127,"La Victoria Green Taco Sauce Mild, 12 Ounce (Pack of 6)",72.00,Fl Oz,fl_oz,91,18,4,6,3,0,0,7,6,1,1,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,12.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2129.294128,340.194277,NaN,6,2041.165665,NaN,6.0,4.290459,4.521789,2.944439,1.386294,1.945910,0.693147,NaN,NaN,NaN,NaN,7.664015,5.832452,NaN,7.621766,NaN,1.945910,4.89
1,198967,"Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)",32.00,Ounce,ounce,511,80,9,17,11,5,0,20,4,1,1,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,8.00,NaN,NaN,NaN,NaN,NaN,NaN,907.184740,NaN,226.796185,NaN,4,907.184740,NaN,4.0,3.496508,6.238325,4.394449,2.484907,1.609438,0.693147,NaN,NaN,NaN,6.811448,NaN,5.428451,NaN,6.811448,NaN,1.609438,13.12
2,261251,"Bear Creek Hearty Soup Bowl, Creamy Chicken with Rice, 1.9 Ounce (Pack of 6)",11.40,Ounce,ounce,328,59,9,12,9,5,0,14,6,1,1,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,1.90,NaN,NaN,0.0,NaN,NaN,NaN,323.184564,NaN,53.864094,NaN,6,323.184564,NaN,6.0,2.517696,5.796058,4.094345,2.302585,1.945910,0.693147,NaN,NaN,NaN,5.781313,NaN,4.004859,NaN,5.781313,NaN,1.945910,1.97
3,55858,"Judee’s Blue Cheese Powder 11.25 oz - Gluten-Free and Nut-Free - Use in Seasonings and Salad Dressings - Great for Dips, Spreads and Sauces - Made in USA",11.25,Ounce,ounce,1318,211,10,17,8,5,1,47,1,1,1,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,11.25,NaN,NaN,NaN,NaN,NaN,NaN,318.932135,NaN,318.932135,NaN,1,318.932135,NaN,1.0,2.505526,7.184629,5.356586,2.197225,0.693147,0.693147,NaN,NaN,NaN,5.768109,NaN,5.768109,NaN,5.768109,NaN,0.693147,30.34
4,292686,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per case.",12.00,Count,count,155,28,5,13,5,1,0,13,1,12,1,NaN,NaN,NaN,0,1,0,0,0,0,0,0,0,0,12.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,360.038944,NaN,12,4320.467324,NaN,12.0,2.564949,5.049856,3.367296,1.791759,0.693147,2.564949,NaN,NaN,NaN,NaN,NaN,5.888986,NaN,8.371350,NaN,2.564949,66.49


## 10. Align with the Permanent V0.3 Split

In [10]:
train_features = train_features_all[
    train_features_all['sample_id'].isin(set(train_ids))
].copy()

valid_features = train_features_all[
    train_features_all['sample_id'].isin(set(valid_ids))
].copy()

print('Train features rows:', len(train_features))
print('Valid features rows:', len(valid_features))

assert len(train_features) == len(train_ids)
assert len(valid_features) == len(valid_ids)

Train features rows: 60000
Valid features rows: 15000


## 11. Coverage Summary

In [11]:
coverage_cols = [
    'item_name', 'raw_value', 'raw_unit', 'unit_normalized',
    'servings', 'pallet_cases', 'explicit_count',
    'measurement_ounce', 'measurement_fl_oz', 'measurement_pound',
    'measurement_gram', 'measurement_kilogram', 'measurement_ml', 'measurement_liter',
    'raw_weight_grams', 'raw_volume_ml',
    'parsed_weight_grams', 'parsed_volume_ml',
]

coverage_rows = []
for col in coverage_cols:
    if col == 'unit_normalized':
        present = (train_features_all[col] != 'unknown').sum()
    else:
        present = train_features_all[col].notna().sum()
    coverage_rows.append({
        'feature': col,
        'present_rows': int(present),
        'coverage_pct': float(present / len(train_features_all) * 100),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values('coverage_pct', ascending=False)
display(coverage_df)

,feature,present_rows,coverage_pct
2,raw_unit,75000,100.000000
3,unit_normalized,74999,99.998667
0,item_name,74993,99.990667
1,raw_value,74060,98.746667
16,parsed_weight_grams,53695,71.593333
7,measurement_ounce,44784,59.712000
14,raw_weight_grams,44324,59.098667
15,raw_volume_ml,11357,15.142667
6,explicit_count,10862,14.482667
17,parsed_volume_ml,8992,11.989333


## 12. Unit Distribution After Normalization

In [12]:
unit_summary = (
    train_features_all['unit_normalized']
    .value_counts(dropna=False)
    .rename_axis('unit_normalized')
    .reset_index(name='count')
)
unit_summary['pct'] = unit_summary['count'] / len(train_features_all) * 100
display(unit_summary.head(30))
unit_summary.to_csv(FEATURES_DIR / 'v04_unit_summary.csv', index=False)

,unit_normalized,count,pct
0,ounce,44028,58.704000
1,count,18330,24.440000
2,fl_oz,11336,15.114667
3,none,940,1.253333
4,pound,240,0.320000
5,gram,50,0.066667
6,milliliter,14,0.018667
7,liter,7,0.009333
8,kilogram,6,0.008000
9,jar,4,0.005333


## 13. Inspect Quantity Extraction

In [13]:
inspect_cols = [
    'sample_id', 'price', 'pack_quantity', 'case_quantity', 'set_quantity',
    'servings', 'pallet_cases', 'explicit_count', 'effective_count',
    'total_parsed_weight_grams', 'total_parsed_volume_ml'
]

display(
    train_features_all.nlargest(25, 'effective_count')[inspect_cols]
)

,sample_id,price,pack_quantity,case_quantity,set_quantity,servings,pallet_cases,explicit_count,effective_count,total_parsed_weight_grams,total_parsed_volume_ml
6632,297241,85.460,500,500,1,NaN,NaN,NaN,250000.0,1.133981e+06,NaN
46493,95961,45.680,500,500,1,NaN,NaN,NaN,250000.0,1.417476e+06,NaN
10404,44172,16.490,1,1,1,NaN,NaN,93207.0,93207.0,NaN,NaN
49926,262169,12.990,1,1,1,NaN,NaN,84652.0,84652.0,1.063107e+01,NaN
50136,23683,28.840,1,1,1,50000.0,NaN,NaN,50000.0,4.535924e+02,NaN
40906,297284,29.590,200,200,1,NaN,NaN,NaN,40000.0,4.989516e+05,NaN
72352,85139,59.990,200,200,1,NaN,NaN,NaN,40000.0,3.401943e+05,NaN
1552,199552,84.520,192,192,1,NaN,NaN,NaN,36864.0,1.672123e+05,NaN
51451,154377,35.390,1,1,1,NaN,NaN,35180.0,35180.0,1.063107e+04,NaN
17229,23234,21.500,1,3,1,NaN,NaN,24514.0,24514.0,1.437321e+03,NaN


## 14. Compare Structured Features with Price

These correlations are diagnostic only; nonlinear models in V0.5 can exploit interactions far better than simple correlations.

In [14]:
numeric_cols = train_features_all.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in {'sample_id'}]

corr = train_features_all[numeric_cols].corr(method='spearman')['price']
corr = corr.drop('price').sort_values(key=lambda s: s.abs(), ascending=False)

display(corr.head(30).to_frame('spearman_with_price'))
corr.to_csv(METRICS_DIR / 'v04_structured_feature_correlations.csv')

,spearman_with_price
log1p_pallet_cases,1.000000
pallet_cases,1.000000
measurement_kilogram,0.486701
measurement_pound,0.457366
log1p_servings,0.382449
servings,0.382449
parsed_volume_ml,0.368044
log1p_parsed_volume_ml,0.368044
log1p_explicit_count,0.293987
explicit_count,0.293987


## 15. Price by High-Level Commerce Flags

In [15]:
flag_cols = [
    'has_pack', 'has_case', 'has_pallet', 'has_bulk', 'has_serving',
    'has_bottle', 'has_can', 'has_bag', 'has_frozen', 'has_organic'
]

flag_rows = []
for col in flag_cols:
    part = train_features_all[train_features_all[col] == 1]
    flag_rows.append({
        'flag': col,
        'rows': len(part),
        'median_price': part['price'].median() if len(part) else np.nan,
        'mean_price': part['price'].mean() if len(part) else np.nan,
    })

flag_summary = pd.DataFrame(flag_rows).sort_values('rows', ascending=False)
display(flag_summary)
flag_summary.to_csv(FEATURES_DIR / 'v04_flag_summary.csv', index=False)

,flag,rows,median_price,mean_price
0,has_pack,31773,15.580,25.121918
6,has_can,16263,15.500,25.449112
7,has_bag,15782,18.490,26.999636
4,has_serving,10267,18.150,28.305444
9,has_organic,9753,15.895,24.927794
5,has_bottle,8409,20.080,30.628248
3,has_bulk,5261,22.990,35.520820
1,has_case,2881,35.900,56.592117
8,has_frozen,1062,26.945,57.127589
2,has_pallet,41,21.950,49.947561


## 16. Inspect Known V0.3 Failure Patterns

In [16]:
# These patterns were prominent among the largest V0.3 errors.
failure_mask = (
    (train_features_all['has_case'] == 1) |
    (train_features_all['has_pallet'] == 1) |
    (train_features_all['effective_count'] >= 24) |
    (train_features_all['total_parsed_weight_grams'] >= 10000) |
    (train_features_all['total_parsed_volume_ml'] >= 10000)
)

failure_like = train_features_all[failure_mask].copy()

print('Rows matching large-quantity/bulk signals:', len(failure_like))
display(
    failure_like.nlargest(30, 'price')[
        ['sample_id', 'price', 'pack_quantity', 'case_quantity', 'pallet_cases',
         'effective_count', 'total_parsed_weight_grams', 'total_parsed_volume_ml',
         'has_case', 'has_pallet', 'has_bulk']
    ]
)

Rows matching large-quantity/bulk signals: 10292


,sample_id,price,pack_quantity,case_quantity,pallet_cases,effective_count,total_parsed_weight_grams,total_parsed_volume_ml,has_case,has_pallet,has_bulk
58617,229126,2796.00,1,1,NaN,2752.0,NaN,NaN,0,0,0
22839,62251,1188.00,1,1,NaN,33.0,3.900000,NaN,0,0,0
68388,236048,921.50,1,1,NaN,1.0,13607.771100,NaN,1,0,1
72853,262694,779.25,1,1,NaN,432.0,NaN,NaN,0,0,0
73067,190026,739.99,1,1,84.0,2016.0,479.106941,499.792650,1,1,0
45580,272914,691.16,1,16,NaN,18.0,417.304980,NaN,1,0,0
34167,218412,600.59,1,1,NaN,2000.0,NaN,NaN,0,0,1
67184,256003,546.94,1,1,NaN,1.0,15875.732950,NaN,0,0,0
27100,55763,536.41,1,216,NaN,216.0,NaN,12328.613004,1,0,0
49975,177820,469.49,1,1,NaN,5.0,NaN,249.896325,1,0,0


## 17. Build Modeling-Ready Feature Tables

In [17]:
DROP_TEXT_COLUMNS = ['item_name', 'raw_unit']

def modeling_table(df, include_price=False):
    cols = [c for c in df.columns if c not in DROP_TEXT_COLUMNS]
    if not include_price:
        cols = [c for c in cols if c != 'price']
    return df[cols].copy()

train_model_features = modeling_table(train_features, include_price=True)
valid_model_features = modeling_table(valid_features, include_price=True)
test_model_features = modeling_table(test_features, include_price=False)

print('Train modeling table:', train_model_features.shape)
print('Valid modeling table:', valid_model_features.shape)
print('Test modeling table :', test_model_features.shape)

Train modeling table: (60000, 59)
Valid modeling table: (15000, 59)
Test modeling table : (75000, 58)


## 18. Save Feature Artifacts

Parquet is preferred for modeling because it preserves dtypes and loads faster than CSV.

In [18]:
train_model_features.to_parquet(FEATURES_DIR / 'v04_train_features.parquet', index=False)
valid_model_features.to_parquet(FEATURES_DIR / 'v04_validation_features.parquet', index=False)
test_model_features.to_parquet(FEATURES_DIR / 'v04_test_features.parquet', index=False)

# Also keep a compact CSV sample for human inspection.
train_model_features.head(1000).to_csv(
    FEATURES_DIR / 'v04_train_features_sample.csv',
    index=False,
)

print('Saved modeling feature artifacts.')

Saved modeling feature artifacts.


## 19. Save Feature Manifest

In [19]:
feature_manifest = {
    'version': 'V0.4',
    'train_rows': int(len(train_model_features)),
    'validation_rows': int(len(valid_model_features)),
    'test_rows': int(len(test_model_features)),
    'feature_columns': [c for c in train_model_features.columns if c not in {'price'}],
    'categorical_columns': ['unit_normalized'],
    'target_column': 'price',
    'split_source': 'artifacts/splits created in V0.3',
}

with (FEATURES_DIR / 'v04_feature_manifest.json').open('w') as f:
    json.dump(feature_manifest, f, indent=2)

print('Saved:', FEATURES_DIR / 'v04_feature_manifest.json')

Saved: /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion/artifacts/features/v04_feature_manifest.json


## 20. V0.4 Final Summary

In [20]:
print('=' * 82)
print('PRICEFUSION V0.4 — ADVANCED STRUCTURED FEATURE ENGINEERING COMPLETE')
print('=' * 82)
print(f'Train rows                : {len(train_model_features):,}')
print(f'Validation rows           : {len(valid_model_features):,}')
print(f'Test rows                 : {len(test_model_features):,}')
print(f'Model feature count       : {len(feature_manifest["feature_columns"]):,}')
print(f'Normalized unit categories: {train_features_all["unit_normalized"].nunique():,}')
print(f'Rows with parsed weight   : {train_features_all["parsed_weight_grams"].notna().sum():,}')
print(f'Rows with parsed volume   : {train_features_all["parsed_volume_ml"].notna().sum():,}')
print(f'Rows with servings        : {train_features_all["servings"].notna().sum():,}')
print(f'Rows with pallet cases    : {train_features_all["pallet_cases"].notna().sum():,}')
print('=' * 82)
print('Next milestone: PriceFusion V0.5 — CatBoost / LightGBM Structured Modeling')

PRICEFUSION V0.4 — ADVANCED STRUCTURED FEATURE ENGINEERING COMPLETE
Train rows                : 60,000
Validation rows           : 15,000
Test rows                 : 75,000
Model feature count       : 58
Normalized unit categories: 41
Rows with parsed weight   : 53,695
Rows with parsed volume   : 8,992
Rows with servings        : 1,613
Rows with pallet cases    : 2
Next milestone: PriceFusion V0.5 — CatBoost / LightGBM Structured Modeling
